# Jackson et al. (2020) scRNA-seq and proteomics Integration

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.stats import pearsonr

## 1. Load Jackson expression + metadata and set up AnnData

In [ ]:

meta = pd.read_csv(
    "GSM4304272_IL13_chronic_metadata.txt",
    sep="\t",
    index_col=0
)

expr = pd.read_csv(
    "GSM4304272_IL13_chronic_norm_matrix.txt",
    sep="\t",
    index_col=0
)

In [ ]:
adata_chronic = sc.AnnData(
    X=expr.T.values,
    obs=meta.copy(),
    var=pd.DataFrame(index=expr.index.astype(str))
)

adata_chronic.var["gene"] = adata_chronic.var.index
adata_chronic.var_names_make_unique()

adata_chronic.layers["norm_expr"] = adata_chronic.X.copy()

In [ ]:
print(adata_chronic.obs.columns)
print(adata_chronic.obs["treatment"].value_counts())
print(adata_chronic.obs["donor"].value_counts())
print(adata_chronic.obs["clusters"].value_counts().sort_index())

In [ ]:
pd.crosstab(
    adata_chronic.obs["clusters"],
    adata_chronic.obs["treatment"]
)

In [ ]:
adata_chronic.write_h5ad("jackson_GSE145013_chronic.h5ad")

### 2.1 Assign cell populations

In [ ]:
adata_chronic.obs["cell_state"] = "other"

# ciliated control
adata_chronic.obs.loc[
    adata_chronic.obs["clusters"].isin(["c1", "c3"]),
    "cell_state"
] = "ciliated_control"

# ciliated IL13
adata_chronic.obs.loc[
    adata_chronic.obs["clusters"].isin(["c2", "c4"]),
    "cell_state"
] = "ciliated_il13"


In [ ]:
# because there's overlap in secretory (club + goblet = c5-c8) and goblet cells = c7 + c8, just make a new column or it will overwrite the previous mappings
# also do the same for club cells = c5 + c6, so we can compare club vs goblet cells in the future
adata_chronic.obs["cell_state_goblet"] = "other"

# goblet control
adata_chronic.obs.loc[
    adata_chronic.obs["clusters"] == "c7",
    "cell_state_goblet"
] = "goblet_control"

# goblet IL13
adata_chronic.obs.loc[
    adata_chronic.obs["clusters"] == "c8",
    "cell_state_goblet"
] = "goblet_il13"

# club cells
adata_chronic.obs["cell_state_club"] = "other"
adata_chronic.obs.loc[
    adata_chronic.obs["clusters"] == "c5",
    "cell_state_club"
] = "club_control"
adata_chronic.obs.loc[
    adata_chronic.obs["clusters"] == "c6",
    "cell_state_club"
] = "club_il13"

In [ ]:
pd.crosstab(
    adata_chronic.obs["cell_state_club"],
    adata_chronic.obs["treatment"]
)

### 2.2 Calculate RNA expression summaries

In [ ]:

def make_rna_summary_from_group(adata, group_col, group_name):
    sub = adata[adata.obs[group_col] == group_name].copy()
    rna_mean = np.array(sub.X.mean(axis=0)).flatten()
    rna_pct = np.array((sub.X > 0).mean(axis=0)).flatten()
    n_cells_expressing = np.array((sub.X > 0).sum(axis=0)).flatten()
    rna_sum = np.array(sub.X.sum(axis=0)).flatten()
    # divide total expression by the number of cells expressing the gene, but only if the number of expressing cells is not zero
    rna_mean_nonzero = np.zeros_like(rna_sum, dtype=float)
    nonzero = n_cells_expressing > 0
    rna_mean_nonzero[nonzero] = (rna_sum[nonzero] / n_cells_expressing[nonzero])
    
    return pd.DataFrame({
        "gene": sub.var_names.astype(str),
        "jackson_group": group_name,
        "rna_mean": rna_mean,
        "rna_pct_cells": rna_pct,
        "rna_mean_nonzero": rna_mean_nonzero,
        "n_cells_expressing": n_cells_expressing,
        "n_cells_group": sub.n_obs
    })

In [ ]:
adata_expr = adata_chronic.copy()
adata_expr.X = adata_chronic.layers["norm_expr"].copy()
adata_expr.obs["cell_state"] = adata_chronic.obs["cell_state"].copy()
adata_expr.obs["cell_state_goblet"] = adata_chronic.obs["cell_state_goblet"].copy()
adata_expr.obs["cell_state_club"] = adata_chronic.obs["cell_state_club"].copy()

In [ ]:
jackson_ciliated_control_rna_df = make_rna_summary_from_group(
    adata_expr,
    "cell_state",
    "ciliated_control"
)

jackson_ciliated_il13_rna_df = make_rna_summary_from_group(
    adata_expr,
    "cell_state",
    "ciliated_il13"
)

jackson_goblet_control_rna_df = make_rna_summary_from_group(
    adata_expr,
    "cell_state_goblet",
    "goblet_control"
)

jackson_goblet_il13_rna_df = make_rna_summary_from_group(
    adata_expr,
    "cell_state_goblet",
    "goblet_il13"
)

jackson_club_control_rna_df = make_rna_summary_from_group(
    adata_expr,
    "cell_state_club",
    "club_control"
)

jackson_club_il13_rna_df = make_rna_summary_from_group(
    adata_expr,
    "cell_state_club",
    "club_il13"
)

In [ ]:
def rename_rna_columns(df, condition):
    return df.rename(columns={
        "rna_mean": f"rna_mean_{condition}",
        "rna_pct_cells": f"rna_pct_cells_{condition}",
        "rna_mean_nonzero": f"rna_mean_nonzero_{condition}",
        "n_cells_expressing": f"n_cells_expressing_{condition}",
        "n_cells_group": f"n_cells_group_{condition}"
    })

jackson_goblet_control_rna_df = rename_rna_columns(
    jackson_goblet_control_rna_df, "ctrl"
)

jackson_goblet_il13_rna_df = rename_rna_columns(
    jackson_goblet_il13_rna_df, "il13"
)

jackson_club_control_rna_df = rename_rna_columns(
    jackson_club_control_rna_df, "ctrl"
)

jackson_club_il13_rna_df = rename_rna_columns(
    jackson_club_il13_rna_df, "il13"
)

jackson_ciliated_control_rna_df = rename_rna_columns(
    jackson_ciliated_control_rna_df, "ctrl"
)

jackson_ciliated_il13_rna_df = rename_rna_columns(
    jackson_ciliated_il13_rna_df, "il13"
)


## 3. Load and prepare proteomics data

The proteomics input files are not included because the dataset is unpublished

In [ ]:
goblet_protein_df = pd.read_csv("ver_3_goblet_sig_hits_results.csv")
club_protein_df = pd.read_csv("ver_3_secretory_sig_hits_results.csv")
mccs_protein_df = pd.read_csv("ver_3_mccs_sig_hits_results.csv")

### 3.1 Prepare protein-to-gene mappings

In [ ]:
club_protein_df = club_protein_df[[
    "Genes", "Protein.Group", "Protein.Names","Description", "Log2.Mean.ctrl", "Log2.Mean.il13", "Log2.Mean", "Log2.Difference", "p.value", "Rank", "Significance", "ctrl_abundance_bin", "il13_abundance_bin", "mean_abundance_bin", "class"
]].rename(columns={
    "Genes": "gene",
    "Protein.Group": "protein_group",
    "Protein.Names": "protein_names",
    "Description": "protein_description",
    "Log2.Mean.ctrl": "protein_log2_control",
    "Log2.Mean.il13": "protein_log2_il13",
    "Log2.Mean": "protein_log2mean",
    "Log2.Difference": "protein_log2difference",
    "p.value": "protein_pvalue",
    "Rank": "protein_log2rank",
    "Significance": "protein_significance",
    "ctrl_abundance_bin": "protein_ctrl_abundance_bin",
    "il13_abundance_bin": "protein_il13_abundance_bin",
    "mean_abundance_bin": "protein_mean_abundance_bin",
    "class": "protein_class"
})
mccs_protein_df = mccs_protein_df[[
    "Genes", "Protein.Group", "Protein.Names","Description", "Log2.Mean.ctrl", "Log2.Mean.il13", "Log2.Mean", "Log2.Difference", "p.value", "Rank", "Significance", "ctrl_abundance_bin", "il13_abundance_bin", "mean_abundance_bin", "class"
]].rename(columns={
    "Genes": "gene",
    "Protein.Group": "protein_group",
    "Protein.Names": "protein_names",
    "Description": "protein_description",
    "Log2.Mean.ctrl": "protein_log2_control",
    "Log2.Mean.il13": "protein_log2_il13",
    "Log2.Mean": "protein_log2mean",
    "Log2.Difference": "protein_log2difference",
    "p.value": "protein_pvalue",
    "Rank": "protein_log2rank",
    "Significance": "protein_significance",
    "ctrl_abundance_bin": "protein_ctrl_abundance_bin",
    "il13_abundance_bin": "protein_il13_abundance_bin",
    "mean_abundance_bin": "protein_mean_abundance_bin",
    "class": "protein_class"
})

goblet_protein_df = goblet_protein_df[[
    "Genes", "Protein.Group", "Protein.Names","Description", "Log2.Mean.ctrl", "Log2.Mean.il13", "Log2.Mean", "Log2.Difference", "p.value", "Rank", "Significance", "ctrl_abundance_bin", "il13_abundance_bin", "mean_abundance_bin", "class"
]].rename(columns={
    "Genes": "gene",
    "Protein.Group": "protein_group",
    "Protein.Names": "protein_names",
    "Description": "protein_description",
    "Log2.Mean.ctrl": "protein_log2_control",
    "Log2.Mean.il13": "protein_log2_il13",
    "Log2.Mean": "protein_log2mean",
    "Log2.Difference": "protein_log2difference",
    "p.value": "protein_pvalue",
    "Rank": "protein_log2rank",
    "Significance": "protein_significance",
    "ctrl_abundance_bin": "protein_ctrl_abundance_bin",
    "il13_abundance_bin": "protein_il13_abundance_bin",
    "mean_abundance_bin": "protein_mean_abundance_bin",
    "class": "protein_class"
})

Note: Secretory cells in the proteomics dataset refers to club cells in the Jackson et al. (2020) scrnaseq dataset

In [ ]:
club_protein_full = club_protein_df.copy()
club_protein_clean = club_protein_df[~club_protein_df["gene"].str.contains(";", na=False)].copy()
mccs_protein_full = mccs_protein_df.copy()
mccs_protein_clean = mccs_protein_df[~mccs_protein_df["gene"].str.contains(";", na=False)].copy()
goblet_protein_full = goblet_protein_df.copy()
goblet_protein_clean = goblet_protein_df[~goblet_protein_df["gene"].str.contains(";", na=False)].copy()

### 3.2 Select a representative protein group for duplicated genes

In [ ]:
# if more than one protein group maps to the same gene symbol, keep the one with the highest control protein abundance

club_protein_clean_rep = (
    club_protein_clean
    .sort_values("protein_log2_control", ascending=False)
    .drop_duplicates(subset="gene", keep="first")
    .copy()
)
mccs_protein_clean_rep = (
    mccs_protein_clean
    .sort_values("protein_log2_control", ascending=False)
    .drop_duplicates(subset="gene", keep="first")
    .copy()
)
goblet_protein_clean_rep = (
    goblet_protein_clean
    .sort_values("protein_log2_control", ascending=False)
    .drop_duplicates(subset="gene", keep="first")
    .copy()
)

### 3.3 Resolving ambiguous protein groups (one protein group -> multiple possible genes)

Some DIA-MS protein groups mapped to more than one gene. These protein groups were first split into their candidate genes and matched to control RNA expression. Where one candidate showed substantially higher RNA expression than the alternatives, the protein group was assigned to that gene. Otherwise, the protein group was retained as ambiguous.

With protein groups called such as H3-3A;H3C1;H3C15, to try to decide which RNA gene should receive the protein measurement, the logic here is:

* split the ambiguous protein group into candidate genes;
* look up RNA abundance for each candidate;
* if no candidate has RNA → leave unresolved;
* if only one candidate has RNA → assign to that one;
* if multiple candidates have RNA:
    * if one is ≥5× the next and ≥0.1 → assign to dominant gene;
    * otherwise retain multiple candidates

In other words...

For ambiguous protein groups mapping to multiple gene symbols, RNA expression is used only where one candidate gene showed a clear dominant signal. A five-fold difference (≥5×) over the next highest candidate was chosen as a conservative threshold to avoid assigning a shared protein group based on small/negligible differences in RNA expression. A minimum mean RNA expression of 0.1 was additionally required so that dominance is not just driven by very low or near-zero expression values.

In [3]:
def make_ambiguous_gene_table(protein_df, rna_df, celltype):
    # keep only protein groups containing ;
    ambig = protein_df[protein_df["gene"].notna() & protein_df["gene"].str.contains(";", na=False)].copy()

    # keep original protein group gene string
    ambig["protein_gene_group"] = ambig["gene"]

    # split into separate genes
    ambig["gene"] = ambig["gene"].str.split(";")
    ambig = ambig.explode("gene").copy()

    # remove spaces if any
    ambig["gene"] = ambig["gene"].str.strip()

    # merge each split gene with its RNA expression
    out = ambig.merge(
        rna_df,
        on="gene",
        how="left",
        suffixes=("", f"_{celltype}_rna")
    )

    out["cell_type_protein"] = celltype

    return out

In [ ]:
jackson_ciliated_ambiguous = make_ambiguous_gene_table(
    mccs_protein_full,
    jackson_ciliated_control_rna_df,
    "jackson_ciliated_control"
)


jackson_club_ambiguous = make_ambiguous_gene_table(
    club_protein_full,
    jackson_club_control_rna_df,
    "jackson_club_control"
)



jackson_goblet_ambiguous = make_ambiguous_gene_table(
    goblet_protein_full,
    jackson_goblet_control_rna_df,
    "jackson_goblet_control"
)


In [ ]:
def resolve_ambiguous_protein_groups(df):
    df = df.copy()
    summaries = []
    for group_name, g in df.groupby("protein_gene_group"):
        # the protein gene group
        g = g.copy()
        # count candidate genes e.g. P60709;P63261 is 2
        n_candidates = len(g)
        # count how many have RNA mean
        n_with_rna = g["rna_mean_ctrl"].notna().sum()
        n_missing_rna = g["rna_mean_ctrl"].isna().sum()
        # default outputs
        assigned_genes = [] # list of genes that will receive the protein value
        resolved_gene = np.nan # summary of the assigned gene or genes
        resolution_rule = "unresolved" # why the decision was mad e.g. 5x_dominant, partial_rna_single_candidate, shared_similar_rna
        ambiguous_flag = True

        # genes with RNA data only
        g_rna = g[g["rna_mean_ctrl"].notna()].copy()

        # ------------------------
        # Case 0: no RNA
        # ------------------------
        if n_with_rna == 0:
            assigned_genes = []
            resolved_gene = np.nan
            resolution_rule = "no_rna_data"

        # ------------------------
        # Case 1: only one RNA gene
        # ------------------------
        elif n_with_rna == 1: # e.g. GATD3 = 0.7, GATD3B = NaN -> so n_with_rna == 1 is TRUE
            assigned_genes = g_rna["gene"].tolist()
            resolved_gene = assigned_genes[0] # this stores it as a single value e.g. GATD3
            resolution_rule = "partial_rna_single_candidate" # only one candidate gene had RNA data, so protein is tentatively assigned to that gene

        # ------------------------
        # Case 2: multiple RNA genes - all or at least 2 candidates have RNA data
        # ------------------------
        else:
            # sort by increasing RNA mean e.g. H3-3A = 2.157, H3C1 = 0.001, H3C15 = 0.000
            g_sorted = g_rna.sort_values("rna_mean_ctrl", ascending=False)
            # get the top gene and the rna_mean (first row) e.g. top_gene = "H3-3A", top_rna = 2.157
            top_gene = g_sorted["gene"].iloc[0]
            top_rna = g_sorted["rna_mean_ctrl"].iloc[0]
            # get the next top gene and the rna_mean
            # if there is more than one candidate, this gets the second-highest RNA e.g. second_rna = 0.0001
            second_rna = g_sorted["rna_mean_ctrl"].iloc[1]
            
            # 1. all candidates have RNA mean value (complete)
            # Rule: top gene must be at least 5x higher and > 0.1 than next highest - gets rid of bias due to small number
            if n_with_rna == n_candidates:
                if (top_rna >= 5 * (second_rna + 1e-9)) and (top_rna >= 0.1): # adding 1e-9 avoids errors/issues where second_rna is 0.000, so I just add a tiny value here
                    # e.g. here 2.157 (top_rna) bigger than 0.005 (second_rna calulcation value = 0.001), and bigger than 0.1, so H3-3A is dominant
                    assigned_genes = [top_gene]
                    resolved_gene = top_gene
                    resolution_rule = f"complete_5x_dominant"
                else: # Otherwise RNA is too similar; assign protein value to all genes (when all candidate genes have RNA data, but no gene is 5× higher than the next)
                    assigned_genes = g["gene"].tolist() # e.g. ["MAGOH", "MAGOHB"]
                    resolved_gene = ";".join(assigned_genes) # combines to one string e.g. MAGOH;MAGOHB
                    resolution_rule = "complete_shared_similar_rna" #  carry the protein value forward for all candidates

            # 2. for groups where not all candidates have RNA data (partial) e.g. A = 0.7, B = 0.1, C = NaN
            else:
                if (top_rna >= 5 * (second_rna + 1e-9)) and (top_rna >= 0.1): # adding 1e-9 avoids errors/issues where second_rna is 0.000, so I just add a tiny value here
                    # e.g. here 2.157 (top_rna) ≥ 0.005 (second_rna calulcation value), so H3-3A is dominant
                    assigned_genes = [top_gene]
                    resolved_gene = top_gene
                    resolution_rule = f"partial_5x_dominant"
                else: # Otherwise RNA is too similar; assign protein value to all genes (when all candidate genes have RNA data, but no gene is 5× higher than the next)
                    assigned_genes = g_rna["gene"].tolist() # e.g. ["MAGOH", "MAGOHB"]
                    resolved_gene = ";".join(assigned_genes) # combines to one string e.g. MAGOH;MAGOHB
                    resolution_rule = "partial_shared_similar_rna" #  carry the protein value forward for all candidates

        summaries.append({
            "protein_gene_group": group_name,
            "n_candidates": n_candidates,
            "n_with_rna": n_with_rna,
            "n_missing_rna": n_missing_rna,
            "resolved_gene": resolved_gene,
            "assigned_genes": ";".join(assigned_genes),
            "resolution_rule": resolution_rule,
            "ambiguous_flag": ambiguous_flag
        })
    # -----------------------------------
    # Expanded table for merging
    # -----------------------------------
    expanded_df = summary_df.copy()
    expanded_df["assigned_gene"] = (expanded_df["assigned_genes"].fillna("").str.split(";"))
    expanded_df = expanded_df.explode("assigned_gene")
    expanded_df = expanded_df[expanded_df["assigned_gene"] != ""].copy()
    expanded_df = expanded_df.rename(columns={ "protein_gene_group": "original_protein_gene_group"})
    expanded_df = expanded_df[["original_protein_gene_group","assigned_gene","resolution_rule","ambiguous_flag"]].copy()

    return expanded_df

In [ ]:
jackson_ciliated_ambiguous_expanded = resolve_ambiguous_protein_groups(jackson_ciliated_ambiguous)

jackson_club_ambiguous_expanded = resolve_ambiguous_protein_groups(jackson_club_ambiguous)

jackson_goblet_ambiguous_expanded = resolve_ambiguous_protein_groups(jackson_goblet_ambiguous)

### 3.4 Combine unambiguous and resolved protein-gene assignments

In [ ]:
def add_ambiguous_assignments_to_clean_protein(clean_protein_df, full_protein_df, ambiguous_expanded_df):
    clean = clean_protein_df.copy()
    full = full_protein_df.copy()
    expanded = ambiguous_expanded_df.copy()

    # mark clean/unambiguous proteins
    clean["from_ambiguous_protein_group"] = False
    clean["original_protein_gene_group"] = clean["gene"]
    clean["assigned_gene"] = clean["gene"]
    clean["resolution_rule"] = "unambiguous"
    clean["ambiguous_flag"] = False

    # get original ambiguous protein abundance rows
    ambiguous_protein_values = full[full["gene"].str.contains(";", na=False)].copy()
    ambiguous_protein_values = ambiguous_protein_values.rename(columns={"gene": "original_protein_gene_group"})

    # attach protein values to assigned genes
    ambig_assigned = expanded.merge(ambiguous_protein_values,on="original_protein_gene_group",how="left")

    # now assigned_gene becomes the actual gene for downstream merging
    ambig_assigned["gene"] = ambig_assigned["assigned_gene"]
    ambig_assigned["from_ambiguous_protein_group"] = True

    # match clean columns
    keep_cols = clean.columns
    ambig_assigned = ambig_assigned[keep_cols]

    # combine ckean(unambiguous) + resolved/assigned ambiguous tables
    combined = pd.concat([clean, ambig_assigned],ignore_index=True)

    # if same gene appears multiple times, keep highest control abundance
    combined = (
        combined
        .sort_values("protein_log2_control", ascending=False)
        .drop_duplicates(subset="gene", keep="first")
        .copy()
    )

    return combined

In [ ]:
jackson_ciliated_protein_with_ambiguous = add_ambiguous_assignments_to_clean_protein(
    mccs_protein_clean_rep,
    mccs_protein_full,
    jackson_ciliated_ambiguous_expanded
)

jackson_club_protein_with_ambiguous = add_ambiguous_assignments_to_clean_protein(
    club_protein_clean_rep,
    club_protein_full,
    jackson_club_ambiguous_expanded
)

jackson_goblet_protein_with_ambiguous = add_ambiguous_assignments_to_clean_protein(
    goblet_protein_clean_rep,
    goblet_protein_full,
    jackson_goblet_ambiguous_expanded
)

## 4. Integrate proteomics and transcriptomics

### 4.1 Merge protein and RNA measurements by gene

In [ ]:
jackson_ciliated_merged = (
    jackson_ciliated_protein_with_ambiguous
    .merge(jackson_ciliated_control_rna_df, on="gene", how="left")
    .merge(jackson_ciliated_il13_rna_df, on="gene", how="left")
)

jackson_club_merged = (
    jackson_club_protein_with_ambiguous
    .merge(jackson_club_control_rna_df, on="gene", how="left")
    .merge(jackson_club_il13_rna_df, on="gene", how="left")
)

jackson_goblet_merged = (
    jackson_goblet_protein_with_ambiguous
    .merge(jackson_goblet_control_rna_df, on="gene", how="left")
    .merge(jackson_goblet_il13_rna_df, on="gene", how="left")
)

### 4.2 Prepare cell-type-specific integration tables

In [ ]:
def prepare_comparison_table(df, celltype):
    df_out = df.rename(columns={
        "protein_group": f"protein_group_{celltype}",
        "protein_names": f"protein_names_{celltype}",
        "protein_description": f"protein_description_{celltype}",
        "protein_log2_control": f"protein_log2_control_{celltype}",
        "protein_log2_il13": f"protein_log2_il13_{celltype}",
        "protein_log2mean": f"protein_log2_mean_{celltype}",
        "protein_log2difference": f"protein_log2difference_{celltype}",
        "protein_pvalue": f"protein_pvalue_{celltype}",
        "protein_log2rank": f"protein_log2rank_{celltype}",
        "protein_significance": f"protein_significance_{celltype}",
        "protein_ctrl_abundance_bin": f"protein_ctrl_abundance_bin_{celltype}",
        "protein_il13_abundance_bin": f"protein_il13_abundance_bin_{celltype}",
        "protein_mean_abundance_bin": f"protein_mean_abundance_bin_{celltype}",
        "protein_class": f"protein_class_{celltype}",
        "original_protein_gene_group": f"original_protein_gene_group_{celltype}",
        "assigned_gene": f"assigned_gene_{celltype}",
        "resolution_rule": f"resolution_rule_{celltype}",
        "ambiguous_flag": f"ambiguous_flag_{celltype}",
        "rna_mean_ctrl": f"rna_mean_ctrl_{celltype}",
        "rna_pct_cells_ctrl": f"rna_pct_cells_ctrl_{celltype}",
        "rna_mean_nonzero_ctrl": f"rna_mean_nonzero_ctrl_{celltype}",
        "rna_mean_il13": f"rna_mean_il13_{celltype}",
        "rna_pct_cells_il13": f"rna_pct_cells_il13_{celltype}",
        "rna_mean_nonzero_il13": f"rna_mean_nonzero_il13_{celltype}"
    }).copy()

    keep_cols = [
        "gene",
        f"protein_group_{celltype}",
        f"protein_names_{celltype}",
        f"protein_description_{celltype}",
        f"protein_log2_control_{celltype}",
        f"protein_log2_il13_{celltype}",
        f"protein_log2difference_{celltype}",
        f"protein_pvalue_{celltype}",
        f"protein_log2rank_{celltype}",
        f"protein_significance_{celltype}",
        f"protein_ctrl_abundance_bin_{celltype}",
        f"protein_il13_abundance_bin_{celltype}",
        f"protein_mean_abundance_bin_{celltype}",
        f"protein_class_{celltype}",
        f"original_protein_gene_group_{celltype}",
        f"assigned_gene_{celltype}",
        f"resolution_rule_{celltype}",
        f"ambiguous_flag_{celltype}",
        f"rna_mean_ctrl_{celltype}",
        f"rna_pct_cells_ctrl_{celltype}",
        f"rna_mean_nonzero_ctrl_{celltype}",
        f"rna_mean_il13_{celltype}",
        f"rna_pct_cells_il13_{celltype}",
        f"rna_mean_nonzero_il13_{celltype}"
    ]

    keep_cols = [c for c in keep_cols if c in df_out.columns]

    return df_out[keep_cols]


In [ ]:
ciliated_il13_comparison = prepare_comparison_table(jackson_ciliated_merged, "jackson_ciliated")

club_il13_comparison = prepare_comparison_table(jackson_club_merged, "jackson_club")

goblet_il13_comparison = prepare_comparison_table(jackson_goblet_merged, "jackson_goblet")

## 5. Add differential expression results

Start IL13 response analysis (RNA vs protein -> scrnaseq: DESEQ2 logFC vs scproteomics: log2difference)

### 5.1 Jackson et al. (2020) published differential expression results

In [ ]:
# this is taken from the Jackson et al. (2020) published paper's DEG list

jackson_deg = pd.read_excel(
    "jackson_original_DEG_list.xlsx",
    sheet_name="t4_sc.treatmentDEGs_11d"
)

In [ ]:
def make_jackson_deg_direction(jackson_deg, pos_comp, neg_comp):
    up = jackson_deg[
        jackson_deg["comparison"] == pos_comp
    ].copy()

    down = jackson_deg[
        jackson_deg["comparison"] == neg_comp
    ].copy()

    up["jackson_deg_direction"] = "Up"
    down["jackson_deg_direction"] = "Down"

    out = pd.concat([up, down], ignore_index=True)

    return out

In [ ]:
def merge_with_jackson_deg(
    comparison_df,
    jackson_deg,
    pos_comp,
    neg_comp
):
    jackson_cell_deg = make_jackson_deg_direction(
        jackson_deg,
        pos_comp=pos_comp,
        neg_comp=neg_comp
    )

    comparison_df = comparison_df.reset_index(drop=True)
    jackson_cell_deg = jackson_cell_deg.reset_index(drop=True)

    merged = comparison_df.merge(
        jackson_cell_deg,
        on="gene",
        how="left",
        suffixes=("", "_jackson_original")
    )

    merged["in_jackson_original_deg"] = (
        merged["jackson_deg_direction"].notna()
    )

    merged["jackson_sig"] = (
        merged["p_adj_FDR"] < 0.05
    )

    return merged


In [ ]:
# merge ciliated, secretory, goblet with jackson original deg list
ciliated_vs_jackson = merge_with_jackson_deg(
    comparison_df=ciliated_il13_comparison,
    jackson_deg=jackson_deg,
    pos_comp="08_CilIL13_vs_CilCtrl",
    neg_comp="07_CilCtrl_vs_CilIL13"
)
club_vs_jackson = merge_with_jackson_deg(
    comparison_df=club_il13_comparison,
    jackson_deg=jackson_deg,
    pos_comp="04_SecIL13_c6_vs_SecCtrl_c5",
    neg_comp="03_SecCtrl_c5_vs_SecIL13_c6"
)
goblet_vs_jackson = merge_with_jackson_deg(
    comparison_df=goblet_il13_comparison,
    jackson_deg=jackson_deg,
    pos_comp="06_SecIL13_c8_vs_SecCtrl_c7",
    neg_comp="05_SecCtrl_c7_vs_SecIL13_c8"
)

### 5.2 Pseudobulk DESeq2 differential expression results (my own DESeq2 DEG results)

In [ ]:
# -----------------------------
# Load pseudobulk DESeq2 results
# -----------------------------

pb = pd.read_csv("jackson_pseudobulk_deseq2_all_celltypes.csv")

# -----------------------------
# Add DESeq2 significance and direction flags
# -----------------------------

pb["rna_pseudobulk_sig"] = (
    pb["padj"] < 0.05
)
pb["rna_pseudobulk_direction"] = np.select(
    [
        (pb["rna_pseudobulk_sig"]) & (pb["log2FoldChange"] > 0),
        (pb["rna_pseudobulk_sig"]) & (pb["log2FoldChange"] < 0)
    ],
    [
        "Up",
        "Down"
    ],
    default="Not significant"
)
# Rename DESeq2 columns clearly before merging
pb = pb.rename(
    columns={
        "baseMean": "rna_pseudobulk_baseMean",
        "log2FoldChange": "rna_pseudobulk_log2FC",
        "lfcSE": "rna_pseudobulk_lfcSE",
        "stat": "rna_pseudobulk_stat",
        "pvalue": "rna_pseudobulk_pvalue",
        "padj": "rna_pseudobulk_padj"
    }
)
# -----------------------------
# Split by cell type
# -----------------------------
pb_ciliated = pb[pb["celltype"] == "ciliated"].copy()
pb_club = pb[pb["celltype"] == "club"].copy()
pb_goblet = pb[pb["celltype"] == "goblet"].copy()
pb_cols = [
    "gene",
    "rna_pseudobulk_baseMean",
    "rna_pseudobulk_log2FC",
    "rna_pseudobulk_lfcSE",
    "rna_pseudobulk_stat",
    "rna_pseudobulk_pvalue",
    "rna_pseudobulk_padj",
    "rna_pseudobulk_sig",
    "rna_pseudobulk_direction"
]
# -----------------------------
# Merge into RNA-protein integration tables
# -----------------------------
ciliated_vs_jackson = ciliated_vs_jackson.merge(
    pb_ciliated[pb_cols],
    on="gene",
    how="left"
)
club_vs_jackson = club_vs_jackson.merge(
    pb_club[pb_cols],
    on="gene",
    how="left"
)
goblet_vs_jackson = goblet_vs_jackson.merge(
    pb_goblet[pb_cols],
    on="gene",
    how="left"
)
# -----------------------------
# Add Jackson vs pseudobulk agreement flags
# -----------------------------
def add_rna_confidence_flags(df):
    df = df.copy()
    df["rna_high_confidence"] = (
        (df["jackson_sig"] == True) &
        (df["rna_pseudobulk_sig"] == True)
    )
    return df
ciliated_vs_jackson = add_rna_confidence_flags(ciliated_vs_jackson)
club_vs_jackson = add_rna_confidence_flags(club_vs_jackson)
goblet_vs_jackson = add_rna_confidence_flags(goblet_vs_jackson)

## 6. RNA-protein concordance analysis

### 6.1 Assign RNA-protein concordance categories

Direction consistency check: For genes significant in the Jackson et al. analysis but not in the pseudobulk analysis, the direction of the pseudobulk log2FC was checked against the published Jackson direction. No opposite-direction cases were found in goblet (n=589), club (n=418), or multiciliated (n=368) cells.

In [ ]:
def add_concordance_flags(df, celltype):
    """
    Add concordance flags for RNA and protein directionality.
    RNA responsiveness is significance-based. A gene is considered
    RNA-responsive if it is significant in EITHER:
    - the Jackson published DEG analysis (`jackson_sig`), or
    - the donor-level pseudobulk DESeq2 reanalysis
      (`rna_pseudobulk_sig`).
    For all RNA-responsive genes, RNA direction is determined using the sign
    of the pseudobulk DESeq2 log2 fold change:
    - log2FC > 0: Up
    - log2FC < 0: Down
    Genes that are not significant in either RNA analysis are classified as
    RNA No_change, regardless of the pseudobulk log2FC.
    Protein direction is based on protein significance and the sign of the
    protein log2 fold change.
    Concordance categories:
    - Concordant Up:                    RNA Up, Protein Up
    - Concordant Down:                  RNA Down, Protein Down
    - Complete Discordant:              RNA Up/Down, Protein opposite
    - Partial Discordant Protein-only:  RNA No_change, Protein Up or Down
    - Partial Discordant RNA-only:      Protein No_change, RNA Up or Down
    - Both No change:                   RNA No_change, Protein No_change
    """

    df = df.copy()

    # -----------------------------
    # RNA direction: significance from Jackson OR pseudobulk, direction from pseudobulk log2FC
    # -----------------------------

    jackson_sig = (
        df["jackson_sig"]
        .fillna(False)
        .astype(bool)
    )
    pseudo_sig = (
        df["rna_pseudobulk_sig"]
        .fillna(False)
        .astype(bool)
    )
    # Use one consistent RNA effect estimate for direction
    rna_log2fc = pd.to_numeric(
        df["rna_pseudobulk_log2FC"],
        errors="coerce"
    )
    df["rna_direction"] = "No_change"
    rna_overall_sig = jackson_sig | pseudo_sig
    df.loc[
        rna_overall_sig & (rna_log2fc > 0),
        "rna_direction"
    ] = "Up"
    df.loc[
        rna_overall_sig & (rna_log2fc < 0),
        "rna_direction"
    ] = "Down"

    # -----------------------------
    # Protein direction: significance + FC sign
    # -----------------------------
    df["protein_direction"] = "No_change"

    df.loc[
        (df[f"protein_significance_{celltype}"] == "Significant") &
        (df[f"protein_log2difference_{celltype}"] > 0),
        "protein_direction"
    ] = "Up"

    df.loc[
        (df[f"protein_significance_{celltype}"] == "Significant") &
        (df[f"protein_log2difference_{celltype}"] < 0),
        "protein_direction"
    ] = "Down"

    # -----------------------------
    # Concordance categories
    # -----------------------------
    df["concordance"] = np.select(
        [
            (df["rna_direction"] == "Up") &
            (df["protein_direction"] == "Up"),

            (df["rna_direction"] == "Down") &
            (df["protein_direction"] == "Down"),

            (df["rna_direction"] == "Up") &
            (df["protein_direction"] == "Down"),

            (df["rna_direction"] == "Down") &
            (df["protein_direction"] == "Up"),

            (df["rna_direction"] == "No_change") &
            (df["protein_direction"].isin(["Up", "Down"])),

            (df["protein_direction"] == "No_change") &
            (df["rna_direction"].isin(["Up", "Down"])),

            (df["rna_direction"] == "No_change") &
            (df["protein_direction"] == "No_change")
        ],
        [
            "Concordant Up",
            "Concordant Down",
            "Complete Discordant",
            "Complete Discordant",
            "Partial Discordant Protein-only",
            "Partial Discordant RNA-only",
            "Both No change"
        ],
        default="Other"
    )

    return df


In [ ]:
ciliated_vs_jackson = add_concordance_flags(ciliated_vs_jackson, "jackson_ciliated")
club_vs_jackson     = add_concordance_flags(club_vs_jackson,     "jackson_club")
goblet_vs_jackson   = add_concordance_flags(goblet_vs_jackson,   "jackson_goblet")

### 6.2 Calculate candidate prioritisation metrics/scoring

In [ ]:
# -----------------------------
# Add protein significance, z-scores and rankings - this is for downstream candidate prioritisation ranking/filtering
# -----------------------------

def add_significance_and_zscores(df,rna_col, protein_col,protein_sig_col):
    df = df.copy()

    df["protein_sig"] = (
        df[protein_sig_col] == "Significant"
    )

    df["high_confidence"] = (
        (df["rna_high_confidence"] == True) &
        df["protein_sig"]
    )

    tmp = df.dropna(
        subset=[rna_col, protein_col]
    ).copy()

    tmp["rna_z"] = stats.zscore(
        tmp[rna_col],
        nan_policy="omit"
    )

    tmp["protein_z"] = stats.zscore(
        tmp[protein_col],
        nan_policy="omit"
    )

    tmp["combined_z_strength"] = (
        tmp["rna_z"].abs() +
        tmp["protein_z"].abs()
    )

    df = df.drop(
        columns=[
            "rna_z",
            "protein_z",
            "combined_z_strength"
        ],
        errors="ignore"
    )

    df = df.merge(
        tmp[[
            "gene",
            "rna_z",
            "protein_z",
            "combined_z_strength"
        ]],
        on="gene",
        how="left"
    )

    return df

In [ ]:

ciliated_vs_jackson = add_significance_and_zscores(
    df=ciliated_vs_jackson,
    rna_col="rna_pseudobulk_log2FC",
    protein_col="protein_log2difference_jackson_ciliated",
    protein_sig_col="protein_significance_jackson_ciliated"
)
club_vs_jackson = add_significance_and_zscores(
    df=club_vs_jackson,
    rna_col="rna_pseudobulk_log2FC",
    protein_col="protein_log2difference_jackson_club",
    protein_sig_col="protein_significance_jackson_club"
)
goblet_vs_jackson = add_significance_and_zscores(
    df=goblet_vs_jackson,
    rna_col="rna_pseudobulk_log2FC",
    protein_col="protein_log2difference_jackson_goblet",
    protein_sig_col="protein_significance_jackson_goblet"
)


## 7. Quality control and output

In [ ]:

def add_direction_agreement(df):
    """
    Agreement between our significance-based RNA call and Jackson's
    published DEG direction (quality check for check_summary).
    """
    df = df.copy()
    df["direction_agreement"] = np.select(
        [
            (df["rna_direction"] == "Up")   & (df["jackson_deg_direction"] == "Up"),
            (df["rna_direction"] == "Down") & (df["jackson_deg_direction"] == "Down"),
            (df["rna_direction"].isin(["Up", "Down"])) &
            (df["jackson_deg_direction"].isin(["Up", "Down"])) &
            (df["rna_direction"] != df["jackson_deg_direction"]),
        ],
        ["Same_up", "Same_down", "Opposite"],
        default="Not_in_Jackson_or_no_change",
    )
    return df

ciliated_vs_jackson = add_direction_agreement(ciliated_vs_jackson)
club_vs_jackson     = add_direction_agreement(club_vs_jackson)
goblet_vs_jackson   = add_direction_agreement(goblet_vs_jackson)


In [ ]:

# -----------------------------
# Check counts - QC
# -----------------------------

def check_summary(df, celltype, rna_col, protein_col):
    matched = df.dropna(
        subset=[rna_col, protein_col]
    ).copy()

    degs_only = matched[
        matched["in_jackson_original_deg"]
    ].copy()

    print("\n" + celltype.upper())
    print("Matched RNA-protein genes:", len(matched))
    print("Matched genes in Jackson DEG list:", matched["in_jackson_original_deg"].sum())
    print("\nConcordance categories:")
    print(matched["concordance"].value_counts())
    print("\nDirection agreement among Jackson DEGs:")
    print(degs_only["direction_agreement"].value_counts())

In [ ]:
check_summary(
    ciliated_vs_jackson,
    "ciliated",
    "rna_pseudobulk_log2FC",
    "protein_log2difference_jackson_ciliated"
)

In [ ]:
check_summary(
    club_vs_jackson,
    "club",
    "rna_pseudobulk_log2FC",
    "protein_log2difference_jackson_club"
)

In [ ]:
check_summary(
    goblet_vs_jackson,
    "goblet",
    "rna_pseudobulk_log2FC",
    "protein_log2difference_jackson_goblet"
)

In [ ]:

# -----------------------------
# Export final merged tables to Excel (lab-navigable column subset)
# -----------------------------

LAB_COLUMNS_ONLY = True

def lab_columns(celltype):
    """
    Organised columns for looking at RNA-protein integration results.
    """
    return [
        # Identification and final classification
        "gene",
        f"protein_description_{celltype}",
        "concordance",

        # Overall confidence flags
        "rna_high_confidence",
        "protein_sig",
        "high_confidence",

        # Protein results
        f"protein_log2difference_{celltype}",
        f"protein_pvalue_{celltype}",
        f"protein_significance_{celltype}",
        f"protein_class_{celltype}",

        # Pseudobulk DESeq2 RNA results
        "rna_pseudobulk_log2FC",
        "rna_pseudobulk_padj",
        "rna_pseudobulk_sig",
        "rna_pseudobulk_direction",

        # Jackson original DEG results
        "p_adj_FDR",
        "jackson_sig",
        "in_jackson_original_deg",
        "jackson_deg_direction",

        # Final directional calls used for concordance
        "rna_direction",
        "protein_direction",

        # Effect-size ranking
        "rna_z",
        "protein_z",
        "combined_z_strength"
    ]


def _trim(table, celltype):
    """
    Keep only lab_columns that are present, in order
    """
    if not LAB_COLUMNS_ONLY:
        return table
    # Some columns are cell-type/data dependent, so only retain columns that are actually present rather than raising an error
    keep = [c for c in lab_columns(celltype) if c in table.columns]
    return table[keep]


def export_concordance(df,output_path,celltype):
    
    """
    Export concordance categories to Excel using category-specific ranking
    Ranking logic:
    - Concordant / complete discordant:
        high-confidence genes first, then combined_z_strength
    - Protein-only:
        Absolute protein log2FC (should all be protein significant)
    - RNA-only:
        RNA high-confidence genes first, then absolute pseudobulk RNA log2FC
    """

    # Split the dataset into the concordance categories used in the RNA-protein integration analysis

    concordant = df[
        df["concordance"].isin([
            "Concordant Up",
            "Concordant Down"
        ])
    ].copy()

    complete_discordant = df[
        df["concordance"] == "Complete Discordant"
    ].copy()

    partial_discordant_protein_only = df[
        df["concordance"] == "Partial Discordant Protein-only"
    ].copy()

    partial_discordant_rna_only = df[
        df["concordance"] == "Partial Discordant RNA-only"
    ].copy()

    both_no_change = df[
        df["concordance"] == "Both No change"
    ].copy()

    # just in case - but there shouldn't be any "Other" genes in the final output
    other = df[
        df["concordance"] == "Other"
    ].copy()

    # sort sheets by category-specific strengths
    # all genes: not split by categoy, but still sort by category and combined z score
    # sort by category, then effect strength
    order = [
        "Concordant Up",
        "Concordant Down",
        "Complete Discordant",
        "Partial Discordant Protein-only",
        "Partial Discordant RNA-only",
        "Both No change",
        "Other"
    ]

    # Sort genes by following the assigned categories order above instead of alphabetical

    all_genes = df.copy()
    all_genes["concordance"] = pd.Categorical(
        all_genes["concordance"],
        categories=order,
        ordered=True
    )
    # Genes with strongest combined RNA-protein response/effect goes first
    all_genes = all_genes.sort_values(
        by=[
            "concordance",
            "combined_z_strength"
        ],
        ascending=[
            True,
            False
        ],
        na_position="last"
    )
    # For concordant: high-confidence genes first, then strongest combined effect
    concordant = concordant.sort_values(
        by=[
            "high_confidence",
            "combined_z_strength"
        ],
        ascending=[
            False,
            False
        ],
        na_position="last"
    )
    # Complete discordant: high-confidence genes first, then strongest combined effect
    complete_discordant = complete_discordant.sort_values(
        by=[
            "high_confidence",
            "combined_z_strength"
        ],
        ascending=[
            False,
            False
        ],
        na_position="last"
    )
    # Partial discordant protein only: protein-only: protein-significant genes first, then strongest protein effect
    partial_discordant_protein_only = partial_discordant_protein_only.sort_values(
        by=[
            f"protein_log2difference_{celltype}"
        ],
        ascending=[
            False
        ],
        # Just to make sure I'm sorting by absolute effect size for the log2diff column, not the protein_sig boolean column
        key=lambda value: (
            value.abs()
            if value.name == f"protein_log2difference_{celltype}"
            else value
        ),
        na_position="last"
    )
    # Partial discordant rna only: rna-only: rna-significant genes first, then strongest rna effect
    partial_discordant_rna_only = partial_discordant_rna_only.sort_values(
        by=[
            "rna_high_confidence",
            "rna_pseudobulk_log2FC"
        ],
        ascending=[
            False,
            False
        ],
        key=lambda value: (
            value.abs()
            if value.name == "rna_pseudobulk_log2FC"
            else value
        ),
        na_position="last"
    )

    # Export into excel with sheets separated by category
    with pd.ExcelWriter(output_path) as writer:
        _trim(all_genes, celltype).to_excel(
            writer, sheet_name="All_genes", index=False)
        _trim(concordant, celltype).to_excel(
            writer, sheet_name="Concordant", index=False)
        _trim(complete_discordant, celltype).to_excel(
            writer, sheet_name="Complete_Discordant", index=False)
        _trim(partial_discordant_protein_only, celltype).to_excel(
            writer, sheet_name="Protein_only", index=False)
        _trim(partial_discordant_rna_only, celltype).to_excel(
            writer, sheet_name="RNA_only", index=False)
        _trim(both_no_change, celltype).to_excel(
            writer, sheet_name="Both_No_change", index=False)
        _trim(other, celltype).to_excel(
            writer, sheet_name="Other", index=False)
    
    print(f"Saved to {output_path}")

In [ ]:
# Quick check:

for name, table, celltype in [
    ("Ciliated", ciliated_vs_jackson, "jackson_ciliated"),
    ("Goblet", goblet_vs_jackson, "jackson_goblet"),
    ("Club", club_vs_jackson, "jackson_club")
]:
    missing = [
        column
        for column in lab_columns(celltype)
        if column not in table.columns
    ]

    print(f"\n{name}")
    print("Missing columns:", missing)
    print("Other rows:", (table["concordance"] == "Other").sum())

### 7.1 Export integration results

In [ ]:
export_concordance(
    ciliated_vs_jackson,
    "ver_10_jackson_ciliated_concordance_full.xlsx",
    celltype="jackson_ciliated"
)

export_concordance(
    goblet_vs_jackson,
    "ver_10_jackson_goblet_concordance_full.xlsx",
    celltype="jackson_goblet"
)
export_concordance(
    club_vs_jackson,
    "ver_10_jackson_club_concordance_full.xlsx",
    celltype="jackson_club"
)


### 7.2 RNA-protein correlation plots

In [ ]:

def plot_rna_protein_scatter(
    df,
    cell_type,
    rna_col,
    protein_col,
):
    plot_df = df.dropna(
        subset=[rna_col, protein_col]
    ).copy()

    colour_map = {
        "Concordant Up": "#2AD083",
        "Concordant Down": "#008D4E",
        "Complete Discordant": "#D73027",
        "Partial Discordant Protein-only": "#8E44AD",
        "Partial Discordant RNA-only": "#CAA022",
        "Both No change": "#333131"
    }

    label_map = {
        "Concordant Up": "Concordant up",
        "Concordant Down": "Concordant down",
        "Complete Discordant": "Discordant",
        "Partial Discordant Protein-only": "Partial discordant: protein only",
        "Partial Discordant RNA-only": "Partial discordant: RNA only",
        "Both No change": "Both no change"
    }

    plt.figure(figsize=(10, 8))

    plot_order = [
        "Both No change",
        "Partial Discordant RNA-only",
        "Partial Discordant Protein-only",
        "Concordant Up",
        "Concordant Down",
        "Complete Discordant"
    ]

    for category in plot_order:
        group = plot_df[
            plot_df["concordance"] == category
        ]

        if len(group) == 0:
            continue

        # Complete Discordant gets the largest marker, because it's the rarest category -> to be able to spot on the plot.
        
        if category == "Both No change":
            alpha = 0.50
            size = 20
            zorder = 3

        elif category == "Partial Discordant Protein-only":
            alpha = 0.60
            size = 20
            zorder = 3

        elif category == "Partial Discordant RNA-only":
            alpha = 0.80
            size = 20
            zorder = 3

        elif category == "Complete Discordant":
            alpha = 0.85
            size = 20
            zorder = 6

        else:  # Concordant Up / Down
            alpha = 0.8
            size = 20
            zorder = 5

        plt.scatter(
            group[rna_col],
            group[protein_col],
            color=colour_map[category],
            alpha=alpha,
            s=size,
            zorder=zorder,
            edgecolor="none",
            label=label_map[category]
        )

    plt.axhline(0, color="black", lw=0.5)
    plt.axvline(0, color="black", lw=0.5)

    plt.xlabel("RNA log2FC (pseudobulk DESeq2, IL-13 vs Control)", fontsize = 15)
    plt.ylabel("Protein log2FC (IL-13 \u2212 Control)", fontsize = 15)

    plt.title(
        f"{cell_type}: RNA vs Protein response", fontsize = 18
    )

    plt.legend(
        loc="upper right",
        fontsize=14,
        markerscale=1
    )

    plt.tight_layout()
    plt.show()


In [ ]:
plot_rna_protein_scatter(
    df=ciliated_vs_jackson,
    cell_type="Multiciliated",
    rna_col="rna_pseudobulk_log2FC",
    protein_col="protein_log2difference_jackson_ciliated",
)


In [ ]:
plot_rna_protein_scatter(
    df=club_vs_jackson,
    cell_type="Club",
    rna_col="rna_pseudobulk_log2FC",
    protein_col="protein_log2difference_jackson_club",
)


In [ ]:
plot_rna_protein_scatter(
    df=goblet_vs_jackson,
    cell_type="Goblet",
    rna_col="rna_pseudobulk_log2FC",
    protein_col="protein_log2difference_jackson_goblet",
)
